# Solara AI — VGG16 Transfer Learning Training

> **DISCLOSURE:** The Shading class contains synthetically generated images (simulated shadow overlays on clean panel photos), as no source dataset provided real shading examples.

**Dataset:** 4 classes — dust (944), cracks (97), physical_damage (85), shading (350)
**Total:** 1,476 images (train/val/test split)
**Hardware:** Optimized for CPU execution

## Step 0 — Hardware Check

In [ ]:
import os, sys, time, json, datetime
from pathlib import Path

import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

_notebook_dir = Path(__file__).resolve().parent if '__file__' in dir() else Path.cwd()
_model_training_dir = _notebook_dir.parent
sys.path.insert(0, str(_model_training_dir))
sys.path.insert(0, str(_model_training_dir.parent / "app"))

from config import IMG_SIZE, BATCH_SIZE, FAULT_CLASSES, LABEL_MAP, RANDOM_SEED

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Hardware detection
gpus = tf.config.list_physical_devices('GPU')
print("=" * 60)
print("HARDWARE CHECK")
print("=" * 60)
if gpus:
    print(f"GPU DETECTED: {[g.name for g in gpus]}")
    print("Training will use GPU acceleration.")
else:
    print("NO GPU DETECTED — running on CPU")
    print()
    print("Estimated training times for 1,476 images:")
    print("  MobileNetV2 Stage 1 (frozen base): ~3-8 min/epoch")
    print("  MobileNetV2 Stage 2 (fine-tuned):   ~5-12 min/epoch")
    print("  VGG16 Stage 1 (frozen base):        ~8-20 min/epoch")
    print("  VGG16 Stage 2 (fine-tuned):         ~12-30 min/epoch")
    print()
    print("Total estimated times:")
    print("  MobileNetV2: ~30-90 minutes total")
    print("  VGG16: ~60-180 minutes total")
    print()
    print("Note: Times vary heavily by CPU. First epoch is always slower.")
print("=" * 60)
print(f"TensorFlow version: {tf.__version__}")
print(f"Image size: {IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Classes: {FAULT_CLASSES}")

## Shared Setup — Data Loading & Augmentation

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

DATASET_DIR = _model_training_dir / "dataset"
TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "val"

# Load datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    str(TRAIN_DIR),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    seed=RANDOM_SEED,
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    str(VAL_DIR),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    seed=RANDOM_SEED,
    shuffle=False
)

class_names = train_ds.class_names
print(f"Class names from dataset: {class_names}")

# CRITICAL for CPU: cache and prefetch
train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)

print(f"Train batches: {len(train_ds)}")
print(f"Val batches: {len(val_ds)}")

In [ ]:
# Data augmentation (training only)
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
], name="data_augmentation")

print("Augmentation layers:", [l.name for l in data_augmentation.layers])

In [ ]:
# Class weights for imbalanced dataset
train_counts = [660, 67, 59, 245]  # dust, cracks, physical_damage, shading
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(FAULT_CLASSES)),
    y=np.repeat(np.arange(len(FAULT_CLASSES)), train_counts)
)
class_weights = {i: w for i, w in enumerate(class_weights_array)}
print("Class weights:")
for cls, weight in class_weights.items():
    print(f"  {LABEL_MAP[FAULT_CLASSES[cls]]}: {weight:.3f}")

## VGG16 Model Architecture

In [ ]:
from tensorflow.keras.applications import VGG16 as VGG16
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model

# Build model
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base_model.trainable = False  # Freeze for Stage 1

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
output = Dense(len(FAULT_CLASSES), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Model: VGG16")
print(f"Base layers: {len(base_model.layers)}")
print(f"Total parameters: {model.count_params():,}")
print(f"Trainable parameters: {sum(p.numpy().size for p in model.trainable_weights):,}")
model.summary()

## Callbacks

In [ ]:
RESULTS_DIR = _model_training_dir / "results"
MODELS_DIR = _model_training_dir / "models"
RESULTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

# Epoch timer callback
class EpochTimer(keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start = time.time()
    def on_epoch_end(self, epoch, logs=None):
        elapsed = time.time() - self.epoch_start
        logs['epoch_time'] = elapsed
        print(f"  Epoch time: {elapsed:.1f}s")

# Common callbacks
def get_callbacks(stage_name):
    return [
        EpochTimer(),
        keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=4, restore_best_weights=True, verbose=1
        ),
        keras.callbacks.ModelCheckpoint(
            str(MODELS_DIR / f'vgg16_best.keras'),
            monitor='val_accuracy', save_best_only=True, verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1
        ),
    ]

print("Callbacks defined.")

## Stage 1 — Train Head Only (Frozen Base)

- Base model frozen, only new layers train
- Adam(lr=1e-3), max 12 epochs, EarlyStopping(patience=4)

In [ ]:
print("=" * 60)
print(f"STAGE 1: {model_name} — Frozen Base")
print("=" * 60)

stage1_start = time.time()
history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    class_weight=class_weights,
    callbacks=get_callbacks("stage1"),
    verbose=1
)
stage1_time = time.time() - stage1_start
print(f"\nStage 1 completed in {stage1_time/60:.1f} minutes")

## Stage 2 — Fine-Tune Unfrozen Layers

- Unfreeze last 30% of base layers
- Adam(lr=1e-5), max 8 epochs, EarlyStopping(patience=4)

In [ ]:
# Unfreeze last 30% of base layers
base_model.trainable = True
total_base = len(base_model.layers)
unfreeze_from = int(total_base * 0.7)
for layer in base_model.layers[:unfreeze_from]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f"Base model: {total_base} total layers")
print(f"Unfrozen from layer {unfreeze_from}: {trainable_count} trainable")

# Recompile with lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n" + "=" * 60)
print(f"STAGE 2: {model_name} — Fine-Tuning")
print("=" * 60)

stage2_start = time.time()
history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    class_weight=class_weights,
    callbacks=get_callbacks("stage2"),
    verbose=1
)
stage2_time = time.time() - stage2_start
print(f"\nStage 2 completed in {stage2_time/60:.1f} minutes")

In [ ]:
# Merge training histories
def merge_histories(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history.get(key, [])
    return type('History', (), {'history': merged})()

history = merge_histories(history_stage1, history_stage2)
total_epochs = len(history.history['accuracy'])
total_time = stage1_time + stage2_time
print(f"Total epochs trained: {total_epochs}")
print(f"Total training time: {total_time/60:.1f} minutes")
print(f"Average time per epoch: {total_time/total_epochs:.1f}s")

## Training Curves

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(history.history['accuracy'], label='Train', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Val', linewidth=2)
ax1.set_title('VGG16 — Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(history.history['loss'], label='Train', linewidth=2)
ax2.plot(history.history['val_loss'], label='Val', linewidth=2)
ax2.set_title('VGG16 — Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
curves_path = RESULTS_DIR / 'vgg16_training_curves.png'
plt.savefig(str(curves_path), dpi=150, bbox_inches='tight')
plt.close()
print(f"Training curves saved to: {curves_path}")

# Save history JSON
history_path = RESULTS_DIR / 'vgg16_history.json'
with open(str(history_path), 'w') as f:
    json.dump(history.history, f, indent=2)
print(f"History saved to: {history_path}")

## Per-Class Validation Accuracy

In [ ]:
# Evaluate per-class accuracy
from sklearn.metrics import classification_report, confusion_matrix

# Get predictions on validation set
y_true = []
y_pred = []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("=" * 60)
print("PER-CLASS VALIDATION ACCURACY")
print("=" * 60)

# Overall accuracy
overall_acc = np.mean(y_true == y_pred)
print(f"Overall accuracy: {overall_acc:.4f} ({overall_acc*100:.1f}%)")
print()

# Per-class
for i, cls in enumerate(FAULT_CLASSES):
    mask = y_true == i
    if mask.sum() > 0:
        cls_acc = np.mean(y_pred[mask] == i)
        print(f"  {LABEL_MAP[cls]:<20} {mask.sum():>4} samples  acc={cls_acc:.4f} ({cls_acc*100:.1f}%)")
    else:
        print(f"  {LABEL_MAP[cls]:<20}    0 samples  (no val data)")

print()
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=[LABEL_MAP[c] for c in FAULT_CLASSES]))

In [ ]:
# Confusion matrix
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.set(xticks=np.arange(cm.shape[1]), yticks=np.arange(cm.shape[0]),
       xticklabels=[LABEL_MAP[c] for c in FAULT_CLASSES],
       yticklabels=[LABEL_MAP[c] for c in FAULT_CLASSES],
       title=f'{model_name} — Confusion Matrix',
       ylabel='True', xlabel='Predicted')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

# Add text annotations
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black')

plt.colorbar(im)
plt.tight_layout()
cm_path = RESULTS_DIR / f'{model_name.lower()}_confusion_matrix.png'
plt.savefig(str(cm_path), dpi=150, bbox_inches='tight')
plt.close()
print(f"Confusion matrix saved to: {cm_path}")

## Save Final Model

In [ ]:
# Save final model
final_path = MODELS_DIR / f'{model_name.lower()}_final.keras'
model.save(str(final_path))
print(f"Final model saved to: {final_path}")

# Save class indices (for API inference)
class_indices = {name: i for i, name in enumerate(class_names)}
indices_path = MODELS_DIR / 'class_indices.json'
with open(str(indices_path), 'w') as f:
    json.dump(class_indices, f, indent=2)
print(f"Class indices saved to: {indices_path}")
print()
print("CLASS INDICES:")
for name, idx in class_indices.items():
    print(f"  {idx}: {name}")

# Summary
print()
print("=" * 60)
print(f"TRAINING COMPLETE — {model_name}")
print("=" * 60)
print(f"Total epochs: {total_epochs}")
print(f"Total time: {total_time/60:.1f} minutes")
print(f"Best val accuracy: {max(history.history['val_accuracy']):.4f}")
print(f"Final val accuracy: {history.history['val_accuracy'][-1]:.4f}")
print(f"Model file: {final_path}")
print(f"File size: {final_path.stat().st_size / 1024 / 1024:.1f} MB")